<a href="https://colab.research.google.com/github/essensha/cs2-anti-cheat-ml-research/blob/main/Anti_Cheating_(Exisiting_vs_New_Model).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

file_path = "/srv/synthetic_anticheat_telemetry.csv"

# Reload and overwrite the old DataFrame
df = pd.read_csv(file_path, skiprows=1, index_col=False)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(5))

Shape: (12000, 11)
Columns: ['player_id', 'match_id', 'match_day', 'skill_proxy', 'actions_per_min', 'median_reaction_ms', 'timing_regularities', 'target_accuracy', 'repeated_action_ratio', 'synthetic_pattern', 'is_suspicious_simulated']


,player_id,match_id,match_day,skill_proxy,actions_per_min,median_reaction_ms,timing_regularities,target_accuracy,repeated_action_ratio,synthetic_pattern,is_suspicious_simulated
0,100000,204010,27,0.616,151.3,334.8,0.383,0.510,0.066,normal,0
1,100001,200988,18,0.333,143.9,290.8,0.240,0.366,0.026,normal,0
2,100002,204639,52,0.741,155.3,172.5,0.537,0.613,0.421,normal,0
3,100003,203584,31,0.328,127.1,249.7,0.255,0.309,0.146,normal,0
4,100004,204014,19,0.418,129.4,247.0,0.176,0.460,0.204,normal,0


In [ ]:

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n_players = 100_000

players = pd.DataFrame({
    "player_id": np.arange(n_players),
    "skill_proxy": rng.uniform(0, 1, n_players),
    "tenure_days": rng.integers(1, 1500, n_players),
    "region": rng.choice(
        ["NA", "EU", "KR", "BR"],
        size=n_players
    )
})

# Hidden synthetic ground truth
players["true_cheating"] = rng.binomial(
    1, 0.025, n_players
)

# Existing detector is more likely to flag
# cheating players, but misses some.
flag_probability = np.where(
    players["true_cheating"] == 1,
    0.70,
    0.035
)

players["old_model_flag"] = (
    rng.random(n_players) < flag_probability
).astype(int)

# Only flagged players receive investigation labels
players["reviewed"] = players["old_model_flag"]

players["observed_label"] = np.where(
    players["reviewed"] == 1,
    players["true_cheating"],
    np.nan
)

print("Total players:", len(players))
print("True cheating prevalence:",
      round(players["true_cheating"].mean(), 4))
print("Flagged players:",
      players["old_model_flag"].sum())
print("Reviewed players:",
      players["reviewed"].sum())
print("Unlabeled players:",
      players["observed_label"].isna().sum())

print("\nObserved label counts:")
print(players["observed_label"].value_counts(dropna=False))


Total players: 100000
True cheating prevalence: 0.0243
Flagged players: 5150
Reviewed players: 5150
Unlabeled players: 94850

Observed label counts:
observed_label
NaN    94850
0.0     3446
1.0     1704
Name: count, dtype: int64


In [ ]:

population_prevalence = players["true_cheating"].mean()

reviewed_players = players[players["reviewed"] == 1]

reviewed_prevalence = reviewed_players["observed_label"].mean()

print("Population prevalence:",
      round(population_prevalence * 100, 2), "%")

print("Reviewed prevalence:",
      round(reviewed_prevalence * 100, 2), "%")

print("Prevalence ratio:",
      round(reviewed_prevalence / population_prevalence, 2))


Population prevalence: 2.43 %
Reviewed prevalence: 33.09 %
Prevalence ratio: 13.61


Detecting Sample Bias (Selection Bias)

In [ ]:

import numpy as np
import pandas as pd

rng = np.random.default_rng(123)

# Placeholder score, not a trained anomaly model
players["anomaly_score"] = rng.random(len(players))

# 1. Existing model flags
group_1 = players[
    players["old_model_flag"] == 1
].sample(n=1000, random_state=123)

# 2. Unflagged players with highest anomaly scores
group_2 = (
    players[players["old_model_flag"] == 0]
    .nlargest(500, "anomaly_score")
)

# 3. Random players not already selected
selected_ids = pd.concat([
    group_1["player_id"],
    group_2["player_id"]
])

group_3 = players[
    ~players["player_id"].isin(selected_ids)
].sample(n=500, random_state=456)

# Combine cohorts
investigations = pd.concat([
    group_1.assign(sampling_group="existing_flag"),
    group_2.assign(sampling_group="anomaly"),
    group_3.assign(sampling_group="random")
], ignore_index=True)

# Validate allocation
assert len(investigations) == 2000
assert investigations["player_id"].is_unique

# Reveal synthetic ground truth for evaluation
results = (
    investigations.groupby("sampling_group")
    ["true_cheating"]
    .agg(
        investigated="count",
        confirmed="sum",
        cheating_rate="mean"
    )
)

results["cheating_rate"] *= 100

print(results.round(2))


                investigated  confirmed  cheating_rate
sampling_group                                        
anomaly                  500          5            1.0
existing_flag           1000        363           36.3
random                   500         10            2.0


The existing detector produces the highest confirmed-cheating yield, which is expected given its selection mechanism. However, the lower-yield cohorts remain valuable for estimating missed detections and evaluating whether alternative behavioral signals identify suspicious activity outside the existing model's coverage

In [ ]:

rng = np.random.default_rng(2026)

n = len(players)

# Assign two synthetic cheating styles
players["cheat_style"] = "none"

cheater_ids = players.index[
    players["true_cheating"] == 1
]

styles = rng.choice(
    ["fast_reaction", "timing_automation"],
    size=len(cheater_ids),
    p=[0.65, 0.35]
)

players.loc[cheater_ids, "cheat_style"] = styles

# Generate baseline gameplay telemetry
players["reaction_ms"] = rng.normal(260, 40, n)
players["target_accuracy"] = rng.normal(0.55, 0.10, n)
players["timing_regularity"] = rng.normal(0.35, 0.12, n)
players["repeated_action_ratio"] = rng.normal(0.20, 0.08, n)

# Pattern A: fast reaction + high accuracy
fast = players["cheat_style"] == "fast_reaction"

players.loc[fast, "reaction_ms"] -= 95
players.loc[fast, "target_accuracy"] += 0.25

# Pattern B: unusually regular, repetitive actions
automation = players["cheat_style"] == "timing_automation"

players.loc[automation, "timing_regularity"] += 0.45
players.loc[automation, "repeated_action_ratio"] += 0.35

# Keep bounded features in valid ranges
for feature in [
    "target_accuracy",
    "timing_regularity",
    "repeated_action_ratio"
]:
    players[feature] = players[feature].clip(0, 1)

players["reaction_ms"] = players["reaction_ms"].clip(lower=50)

# Inspect behavior by synthetic cheating style
summary = players.groupby("cheat_style")[
    [
        "reaction_ms",
        "target_accuracy",
        "timing_regularity",
        "repeated_action_ratio"
    ]
].mean()

print(summary.round(3))

print("\nCheating style counts:")
print(players["cheat_style"].value_counts())


                   reaction_ms  target_accuracy  timing_regularity  \
cheat_style                                                          
fast_reaction          166.032            0.797              0.354   
none                   260.001            0.550              0.350   
timing_automation      260.053            0.554              0.795   

                   repeated_action_ratio  
cheat_style                               
fast_reaction                      0.198  
none                               0.200  
timing_automation                  0.549  

Cheating style counts:
cheat_style
none                 97569
fast_reaction         1548
timing_automation      883
Name: count, dtype: int64


In [ ]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    confusion_matrix
)

# Simple reaction-time-only baseline
players["reaction_detector_flag"] = (
    players["reaction_ms"] < 190
).astype(int)

# Overall performance against synthetic ground truth
print("Overall precision:",
      round(precision_score(
          players["true_cheating"],
          players["reaction_detector_flag"]
      ), 3))

print("Overall recall:",
      round(recall_score(
          players["true_cheating"],
          players["reaction_detector_flag"]
      ), 3))

# Detection rate within each synthetic cheating style
cheaters = players[
    players["true_cheating"] == 1
]

print("\nDetection rate by cheating style:")
print(
    cheaters.groupby("cheat_style")
    ["reaction_detector_flag"]
    .mean()
    .round(3)
)


Overall precision: 0.226
Overall recall: 0.464

Detection rate by cheating style:
cheat_style
fast_reaction        0.703
timing_automation    0.044
Name: reaction_detector_flag, dtype: float64


We're evaluating improvements to an existing Anti-Cheat detection approach that exhibits uneven coverage across cheating behaviors. Our immediate concerns are limited feature representation and selective investigation labels. We'll benchmark a multifeature classifier, assess performance across cheating patterns, and investigate drift separately as part of model monitoring and robustness testing.

### Stage 2: Multifeature Classifier

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score
)

# Split players before model training
train_idx, test_idx = train_test_split(
    players.index,
    test_size=0.25,
    random_state=42,
    stratify=players["true_cheating"]
)

train = players.loc[train_idx].copy()
test = players.loc[test_idx].copy()

# Train only on cases reviewed by the old detector
labeled_train = train[
    train["observed_label"].notna()
].copy()

feature_sets = {
    "reaction_only": ["reaction_ms"],
    "multifeature": [
        "reaction_ms",
        "target_accuracy",
        "timing_regularity",
        "repeated_action_ratio"
    ]
}

models = {}

for name, features in feature_sets.items():

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

    model.fit(
        labeled_train[features],
        labeled_train["observed_label"].astype(int)
    )

    probabilities = model.predict_proba(
        test[features]
    )[:, 1]

    models[name] = model

    print(f"\n{name}")
    print("ROC-AUC:",
          round(roc_auc_score(
              test["true_cheating"],
              probabilities
          ), 3))
    print("PR-AUC:",
          round(average_precision_score(
              test["true_cheating"],
              probabilities
          ), 3))

    test[f"{name}_prediction"] = (
        probabilities >= 0.5
    ).astype(int)

    print("Recall by cheating style:")

    print(
        test[test["true_cheating"] == 1]
        .groupby("cheat_style")[
            f"{name}_prediction"
        ]
        .mean()
        .round(3)
    )



reaction_only
ROC-AUC: 0.802
PR-AUC: 0.357
Recall by cheating style:
cheat_style
fast_reaction        0.811
timing_automation    0.074
Name: reaction_only_prediction, dtype: float64

multifeature
ROC-AUC: 0.985
PR-AUC: 0.789
Recall by cheating style:
cheat_style
fast_reaction        0.908
timing_automation    0.944
Name: multifeature_prediction, dtype: float64


Quantify the false-positive tradeoff

In [ ]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    confusion_matrix
)

for name in ["reaction_only", "multifeature"]:

    y_true = test["true_cheating"]
    y_pred = test[f"{name}_prediction"]

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_true, y_pred, zero_division=0
    )

    recall = recall_score(
        y_true, y_pred, zero_division=0
    )

    fpr = fp / (fp + tn)

    print(f"\n{name}")
    print("Precision:", round(precision, 3))
    print("Recall:", round(recall, 3))
    print("False-positive rate:", round(fpr, 4))
    print("False positives:", fp)
    print("False positives per 10,000 legitimate players:",
          round(fpr * 10_000, 1))



reaction_only
Precision: 0.17
Recall: 0.549
False-positive rate: 0.0671
False positives: 1636
False positives per 10,000 legitimate players: 670.7

multifeature
Precision: 0.35
Recall: 0.921
False-positive rate: 0.0426
False positives: 1038
False positives per 10,000 legitimate players: 425.5


Brier score: Measures overall probability prediction error. Lower is better.



Calibration by risk band: Compares predicted risk against actual cheating frequency within different score ranges.



Precision by risk band: Measures the proportion of truly suspicious players among those selected for investigation at a particular threshold.

In [ ]:

from sklearn.metrics import brier_score_loss

for name, features in feature_sets.items():

    probabilities = models[name].predict_proba(
        test[features]
    )[:, 1]

    test[f"{name}_probability"] = probabilities

    brier = brier_score_loss(
        test["true_cheating"],
        probabilities
    )

    print(f"{name} Brier score: {brier:.5f}")

# Prevalence-only baseline
prevalence = test["true_cheating"].mean()

baseline_brier = brier_score_loss(
    test["true_cheating"],
    [prevalence] * len(test)
)

print(f"Prevalence baseline Brier: {baseline_brier:.5f}")


reaction_only Brier score: 0.08449
multifeature Brier score: 0.03284
Prevalence baseline Brier: 0.02373


High-risk calibration and Wilson intervals

Let's examine the multifeature model's top 10% highest-risk predictions, divided into five equally sized groups.

In [ ]:

import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportion_confint

# Focus on the 10% highest-risk players
cutoff = test["multifeature_probability"].quantile(0.90)

high_risk = test[
    test["multifeature_probability"] >= cutoff
].copy()

# Split high-risk players into five bands
high_risk["risk_band"] = pd.qcut(
    high_risk["multifeature_probability"],
    q=5,
    duplicates="drop"
)

results = []

for band, group in high_risk.groupby(
    "risk_band", observed=True
):

    n = len(group)
    confirmed = int(group["true_cheating"].sum())

    observed_rate = confirmed / n
    predicted_rate = group[
        "multifeature_probability"
    ].mean()

    ci_low, ci_high = proportion_confint(
        confirmed,
        n,
        alpha=0.05,
        method="wilson"
    )

    results.append({
        "risk_band": str(band),
        "n": n,
        "predicted_rate": predicted_rate,
        "observed_rate": observed_rate,
        "calibration_gap": observed_rate - predicted_rate,
        "wilson_low": ci_low,
        "wilson_high": ci_high
    })

calibration_results = pd.DataFrame(results)

print(calibration_results.round(4).to_string(index=False))


     risk_band   n  predicted_rate  observed_rate  calibration_gap  wilson_low  wilson_high
(0.291, 0.386] 500          0.3379          0.016          -0.3219      0.0081       0.0313
(0.386, 0.531] 500          0.4549          0.034          -0.4209      0.0213       0.0538
(0.531, 0.727] 500          0.6231          0.098          -0.5251      0.0749       0.1272
(0.727, 0.931] 500          0.8323          0.206          -0.6263      0.1729       0.2436
  (0.931, 1.0] 500          0.9807          0.800          -0.1807      0.7627       0.8327


Oh no. The multifeature model may distinguish cheating patterns effectively, but its predicted probabilities are substantially higher than the observed cheating rates. All five bands show overprediction. (predictive rate above rate of Wilson intervals)

Takeaway: This is a plausible explanation for the systematic overprediction: the model learned probabilities from a highly enriched training sample

The multifeature model improves detection coverage but exhibits substantial overprediction across high-risk bands. Given the elevated cheating prevalence among investigator-reviewed training cases, selective labeling is a plausible contributor to this miscalibration. Before deployment, I would investigate the sampling mechanism, recalibrate using representative labeled data, and reassess performance on an independent holdout.

##Test the effect of representative training data

In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score

features = [
    "reaction_ms",
    "target_accuracy",
    "timing_regularity",
    "repeated_action_ratio"
]

# Model A: selectively reviewed cases
reviewed_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

reviewed_model.fit(
    labeled_train[features],
    labeled_train["observed_label"].astype(int)
)

# Model B: representative synthetic sample
representative_train = train.sample(
    n=len(labeled_train),
    random_state=2026
)

representative_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

representative_model.fit(
    representative_train[features],
    representative_train["true_cheating"]
)

for name, model in [
    ("Reviewed-only", reviewed_model),
    ("Representative", representative_model)
]:
    probabilities = model.predict_proba(
        test[features]
    )[:, 1]

    print(f"\n{name}")
    print("Training prevalence:",
          round(
              (labeled_train["observed_label"].mean()
               if name == "Reviewed-only"
               else representative_train["true_cheating"].mean()),
              4
          ))
    print("ROC-AUC:",
          round(roc_auc_score(
              test["true_cheating"],
              probabilities
          ), 4))
    print("Brier score:",
          round(brier_score_loss(
              test["true_cheating"],
              probabilities
          ), 4))
    print("Mean predicted probability:",
          round(probabilities.mean(), 4))



Reviewed-only
Training prevalence: 0.3263
ROC-AUC: 0.9852
Brier score: 0.0328
Mean predicted probability: 0.0908

Representative
Training prevalence: 0.0205
ROC-AUC: 0.9849
Brier score: 0.0097
Mean predicted probability: 0.0241


The two models exhibit nearly identical discrimination, but the representative-data model reduces Brier score by approximately 70%. This suggests that selective investigation labels materially affect probability reliability, even when ranking performance remains strong. Improving the sampling strategy may therefore be more valuable than increasing model complexity

In [ ]:

import pandas as pd
import numpy as np

comparison = test[
    ["true_cheating"]
].copy()

comparison["reviewed_probability"] = (
    reviewed_model.predict_proba(
        test[features]
    )[:, 1]
)

comparison["representative_probability"] = (
    representative_model.predict_proba(
        test[features]
    )[:, 1]
)

for model_name in [
    "reviewed",
    "representative"
]:

    probability_col = f"{model_name}_probability"

    comparison[f"{model_name}_band"] = pd.qcut(
        comparison[probability_col],
        q=10,
        duplicates="drop"
    )

    calibration = (
        comparison.groupby(
            f"{model_name}_band",
            observed=True
        )
        .agg(
            n=("true_cheating", "size"),
            predicted=(probability_col, "mean"),
            observed=("true_cheating", "mean")
        )
    )

    calibration["gap"] = (
        calibration["observed"] -
        calibration["predicted"]
    )

    print(f"\n{model_name.upper()} MODEL")
    print(calibration.round(4))



REVIEWED MODEL
                              n  predicted  observed     gap
reviewed_band                                               
(-0.0009999351, 0.000164]  2500     0.0001    0.0000 -0.0001
(0.000164, 0.000603]       2500     0.0004    0.0000 -0.0004
(0.000603, 0.00152]        2500     0.0010    0.0000 -0.0010
(0.00152, 0.00343]         2500     0.0024    0.0000 -0.0024
(0.00343, 0.00731]         2500     0.0052    0.0000 -0.0052
(0.00731, 0.0151]          2500     0.0107    0.0000 -0.0107
(0.0151, 0.034]            2500     0.0231    0.0008 -0.0223
(0.034, 0.0865]            2500     0.0550    0.0036 -0.0514
(0.0865, 0.292]            2500     0.1644    0.0080 -0.1564
(0.292, 1.0]               2500     0.6458    0.2308 -0.4150

REPRESENTATIVE MODEL
                               n  predicted  observed     gap
representative_band                                          
(-0.00099999517, 1.03e-05]  2500     0.0000    0.0000 -0.0000
(1.03e-05, 3.66e-05]        2500     0.0000 

Training on representative synthetic labels reduced the highest-risk decile's calibration error from 41.5 percentage points to 0.64 percentage points while preserving ROC-AUC. This suggests that selective investigation labels, rather than inadequate model complexity, were a major contributor to probability miscalibration in this experiment. I would prioritize representative investigator audits and independent validation before deployment.

In [ ]:

from statsmodels.stats.proportion import proportion_confint

for name in ["reviewed", "representative"]:

    probability_col = f"{name}_probability"

    cutoff = comparison[probability_col].quantile(0.90)

    top_decile = comparison[
        comparison[probability_col] >= cutoff
    ]

    n = len(top_decile)
    positives = int(top_decile["true_cheating"].sum())

    observed = positives / n
    predicted = top_decile[probability_col].mean()

    low, high = proportion_confint(
        positives,
        n,
        alpha=0.05,
        method="wilson"
    )

    print(f"\n{name.upper()}")
    print("Sample size:", n)
    print("Predicted risk:", round(predicted, 4))
    print("Observed rate:", round(observed, 4))
    print("Wilson 95% CI:",
          (round(low, 4), round(high, 4)))
    print("Predicted rate within CI:",
          low <= predicted <= high)



REVIEWED
Sample size: 2500
Predicted risk: 0.6458
Observed rate: 0.2308
Wilson 95% CI: (0.2147, 0.2477)
Predicted rate within CI: False

REPRESENTATIVE
Sample size: 2500
Predicted risk: 0.2248
Observed rate: 0.2312
Wilson 95% CI: (0.2151, 0.2481)
Predicted rate within CI: True


Turn model scores into an investigation policy

In [ ]:

from sklearn.metrics import precision_score

review_capacity = 200

for name, model in [
    ("Reviewed-only", reviewed_model),
    ("Representative", representative_model)
]:

    scores = model.predict_proba(
        test[features]
    )[:, 1]

    ranked = test.copy()
    ranked["risk_score"] = scores

    top_cases = ranked.nlargest(
        review_capacity,
        "risk_score"
    )

    confirmed = int(
        top_cases["true_cheating"].sum()
    )

    precision_at_capacity = (
        confirmed / review_capacity
    )

    missed = int(
        test["true_cheating"].sum() - confirmed
    )

    print(f"\n{name}")
    print("Cases reviewed:", review_capacity)
    print("Confirmed cheating:", confirmed)
    print("Precision@200:",
          round(precision_at_capacity, 3))
    print("Cheaters not selected:",
          missed)
    print("Lowest selected score:",
          round(top_cases["risk_score"].min(), 4))



Reviewed-only
Cases reviewed: 200
Confirmed cheating: 191
Precision@200: 0.955
Cheaters not selected: 417
Lowest selected score: 0.9937

Representative
Cases reviewed: 200
Confirmed cheating: 191
Precision@200: 0.955
Cheaters not selected: 417
Lowest selected score: 0.8784


Better-calibrated probabilities can support more reliable estimates of expected investigation yield, capacity planning, risk-tier thresholds, and comparisons across cohorts. They should not, by themselves, justify enforcement.

### Stage 4: Production Safety

A model must not produce a cheating-risk score when required telemetry is missing or invalid.

In [ ]:

import numpy as np
import pandas as pd

def score_player(record):
    required = [
        "reaction_ms",
        "target_accuracy",
        "timing_regularity",
        "repeated_action_ratio"
    ]

    if any(f not in record for f in required):
        return {"status": "abstain", "risk_score": None}

    try:
        values = np.array(
            [record[f] for f in required],
            dtype=float
        )
    except (ValueError, TypeError):
        return {"status": "abstain", "risk_score": None}

    if not np.isfinite(values).all():
        return {"status": "abstain", "risk_score": None}

    X = pd.DataFrame([record])[required]
    probability = representative_model.predict_proba(X)[0, 1]

    return {
        "status": "success",
        "risk_score": float(probability)
    }

valid = {
    "reaction_ms": 165,
    "target_accuracy": 0.84,
    "timing_regularity": 0.37,
    "repeated_action_ratio": 0.19
}

missing = {
    "reaction_ms": 165,
    "target_accuracy": 0.84
}

print("Valid:", score_player(valid))
print("Missing:", score_player(missing))


Valid: {'status': 'success', 'risk_score': 0.7159926845241485}
Missing: {'status': 'abstain', 'risk_score': None}


If prediction service crashes: Abstain, log the failure, and use an established fallback process